# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD
<div>
 <h2> CSCI 4253 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
import csv
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf = SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

The two input RDDs are read as raw text. Each record is initially a CSV line stored as a string, so the data must be parsed and converted into useful key-value pairs.


In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

## Parse the input data

I remove the header rows and convert the citation data into integer `(CITING, CITED)` pairs. For the patent data, I keep the original CSV fields so they can be included in the final output.


In [6]:
def parse_csv(line):
    return next(csv.reader([line]))

citations = (
    rddCitations
    .map(parse_csv)
    .filter(lambda row: len(row) >= 2)
    .filter(lambda row: row[0].strip().upper() != "CITING")
    .map(lambda row: (int(row[0]), int(row[1])))
)

patent_rows = (
    rddPatents
    .map(parse_csv)
    .filter(lambda row: len(row) > 5)
    .filter(lambda row: row[0].strip().upper() != "PATENT")
)

print(citations.take(5))
print(patent_rows.take(2))

[(3858241, 956203), (3858241, 1324234), (3858241, 3398406), (3858241, 3557384), (3858241, 3634889)]
[['3070801', '1963', '1096', '', 'BE', '', '', '1', '', '269', '6', '69', '', '1', '', '0', '', '', '', '', '', '', ''], ['3070802', '1963', '1096', '', 'US', 'TX', '', '1', '', '2', '6', '63', '', '0', '', '', '', '', '', '', '', '', '']]


## Create a patent-state lookup

I create a key-value RDD that maps each patent number to its state. Empty state values are kept as `None` because they cannot be used to identify a same-state citation.


In [7]:
patent_states = patent_rows.map(
    lambda row: (int(row[0]), row[5] if len(row) > 5 and row[5] != "" else None)
).cache()

patent_states.take(5)


[(3070801, None),
 (3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA')]

## Find the state of each cited patent

I key the citation data by the cited patent number and join it with the patent-state lookup. This gives the state of each cited patent when that information is available.


In [8]:
citations_by_cited = citations.map(
    lambda x: (x[1], x[0])
)

cited_with_state = (
    citations_by_cited
    .leftOuterJoin(patent_states)
    .map(lambda x: (x[1][0], (x[0], x[1][1])))
    .cache()
)

cited_with_state.take(10)


[(3858252, (2518060, None)),
 (4491990, (2518060, None)),
 (3858279, (3203058, None)),
 (3902226, (3203058, None)),
 (4315350, (3203058, None)),
 (4340329, (3203058, None)),
 (4527985, (3203058, None)),
 (5422152, (3203058, None)),
 (5671971, (3203058, None)),
 (3858349, (2366652, None))]

## Find the state of each citing patent

I join the intermediate RDD with the patent-state lookup again, this time using the citing patent as the key. Each record now contains both the cited and citing states.


In [9]:
citation_states = (
    cited_with_state
    .leftOuterJoin(patent_states)
    .map(lambda x: (x[0], x[1][0][0], x[1][0][1], x[1][1]))
    .cache()
)

citation_states.take(10)


[(4629153, 1782962, None, 'KY'),
 (4629153, 3045962, None, 'KY'),
 (4629153, 1326086, None, 'KY'),
 (4629153, 3090478, 'IL', 'KY'),
 (4629153, 3463436, 'MD', 'KY'),
 (4629153, 1610344, None, 'KY'),
 (4629153, 1208728, None, 'KY'),
 (4629153, 3707272, 'IL', 'KY'),
 (4629153, 2933358, None, 'KY'),
 (4629153, 3734439, 'MI', 'KY')]

## Identify and count same-state citations

I keep only citations for which both states are known and equal. Then I count the matching citations for each citing patent with `reduceByKey`.


In [10]:
same_state_counts = (
    citation_states
    .filter(lambda x: x[2] is not None and x[3] is not None and x[2] == x[3])
    .map(lambda x: (x[0], 1))
    .reduceByKey(operator.add)
    .cache()
)

same_state_counts.take(10)


[(4342026, 1),
 (5374191, 2),
 (5940393, 1),
 (6004914, 3),
 (4923714, 3),
 (3900999, 4),
 (4709766, 1),
 (5622186, 1),
 (5595234, 3),
 (5897181, 1)]

## Add the counts to the patent data

I join the counts back to the original patent records. A patent with no same-state count is assigned zero.


In [11]:
patents_by_id = patent_rows.map(
    lambda row: (int(row[0]), row)
)

augmented_patents = (
    patents_by_id
    .leftOuterJoin(same_state_counts)
    .map(lambda x: x[1][0] + [x[1][1] if x[1][1] is not None else 0])
)

augmented_patents.take(5)


[['3073624',
  '1963',
  '1110',
  '',
  'US',
  'PA',
  '',
  '2',
  '',
  '280',
  '5',
  '55',
  '',
  '3',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0],
 ['3078440',
  '1963',
  '1145',
  '',
  'US',
  'NY',
  '',
  '2',
  '',
  '340',
  '2',
  '21',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0],
 ['3080508',
  '1963',
  '1159',
  '',
  'US',
  'OH',
  '',
  '2',
  '',
  '361',
  '4',
  '45',
  '',
  '3',
  '',
  '0.4444',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0],
 ['3081868',
  '1963',
  '1173',
  '',
  'US',
  'NY',
  '',
  '2',
  '',
  '206',
  '6',
  '68',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0],
 ['3082408',
  '1963',
  '1173',
  '',
  'US',
  'NY',
  '',
  '2',
  '',
  '365',
  '2',
  '24',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0]]

## Top 10 patents by same-state citations

Finally, I sort the augmented patent records by the `SAME_STATE` value in descending order and display the ten patents with the largest counts. The final result is computed using the full citation and patent datasets.


In [12]:
top10 = augmented_patents.takeOrdered(
    10,
    key=lambda row: -row[-1]
)

header = parse_csv(rddPatents.first()) + ["SAME_STATE"]
print(header)
for row in top10:
    print(row)


['PATENT', 'GYEAR', 'GDATE', 'APPYEAR', 'COUNTRY', 'POSTATE', 'ASSIGNEE', 'ASSCODE', 'CLAIMS', 'NCLASS', 'CAT', 'SUBCAT', 'CMADE', 'CRECEIVE', 'RATIOCIT', 'GENERAL', 'ORIGINAL', 'FWDAPLAG', 'BCKGTLAG', 'SELFCTUB', 'SELFCTLB', 'SECDUPBD', 'SECDLWBD', 'SAME_STATE']
['5959466', '1999', '14515', '1997', 'US', 'CA', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', '', 125]
['5983822', '1999', '14564', '1998', 'US', 'TX', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', '', 103]
['6008204', '1999', '14606', '1998', 'US', 'CA', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', '', 100]
['5952345', '1999', '14501', '1997', 'US', 'CA', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', '', 98]
['5958954', '1999', '14515', '1997', 'US', 'CA', '749584', '2', '', '514', '3', '31', '116', '0', '1', '',